In [ ]:
import pandas as pd

In [ ]:
caminho_arquivo = '../data/raw/MiningProcess_Flotation_Plant_Database.csv'

df = pd.read_csv(caminho_arquivo, sep=',', decimal=',', parse_dates=["date"])

In [ ]:
df.shape

In [ ]:
df.dtypes

In [ ]:
df.describe()

In [ ]:
df.head(10)

In [ ]:
# Records per hour
rows_per_hour = df.groupby('date').size()
print(rows_per_hour.value_counts().sort_index())
print(rows_per_hour[rows_per_hour != 180])

In [ ]:
# Missing hours
full_range = pd.date_range(df["date"].min(), df["date"].max(), freq="h")

missing = full_range.difference(rows_per_hour.index)

print(f"expected: {len(full_range)}, present: {len(rows_per_hour)}, missing: {len(missing)}")

missing_s = missing.to_series()
block_id = missing_s.diff().ne(pd.Timedelta(hours=1)).cumsum()
blocks = missing_s.groupby(block_id.values).agg(["min", "max", "count"])
print(blocks)


In [ ]:
# Column that vary

max_distinct_per_hour = df.groupby("date").nunique().max().sort_values()
print(max_distinct_per_hour)

In [ ]:
target = "% Silica Concentrate"
distinct_target = df.groupby("date")[target].nunique()
varying = distinct_target[distinct_target > 1]

print(f"hours with varying target: {len(varying)}")
print(f"from {varying.index.min()} to {varying.index.max()}")

In [ ]:
cols = ['% Iron Concentrate', '% Iron Feed', '% Silica Feed', '% Silica Concentrate', 'Flotation Column 01 Level', 'Flotation Column 03 Air Flow']
mask = (df['date'] >= '2017-04-11 01:00:00') & (df['date'] < '2017-04-21 02:00:00')

s_1 = df.loc[mask]
#diffs = s_1[cols].diff()
diffs = s_1.diff()

print((diffs != 0).sum())           # quantas vezes cada coluna mudou na hora


In [ ]:
diffs.abs().describe()       # tamanho das variações

In [ ]:
step = df.groupby("date")[target].diff()
step_std = step.groupby(df["date"]).std()

print(step_std.loc[varying.index].describe())

In [ ]:
hourly = df.groupby("date")[target].agg(["first", "last"])
next_first = hourly["first"].shift(-1, freq="h").reindex(hourly.index)
gap_to_next = (hourly["last"] - next_first).abs()

print("varying hours:")
print(gap_to_next.loc[varying.index].describe())
print("constant hours (reference):")
print(gap_to_next.drop(varying.index).describe())

In [ ]:
import matplotlib.pyplot as plt

sample_hours = varying.index[[0, len(varying) // 2, -1]]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, hour in zip(axes, sample_hours):
    window = df[df["date"].between(hour - pd.Timedelta(hours=1), hour + pd.Timedelta(hours=1))]
    ax.plot(window[target].to_numpy())
    ax.set_title(str(hour))
    ax.set_xlabel("row (20 s each)")
    ax.set_ylabel(target)
plt.tight_layout()

### Finding: interpolated target values (data leakage risk)

The lab assay for `% Silica Concentrate` is reported once per hour, so within an hour
all 180 rows (20 s each) should carry the same value. In **310 hours (~7.5%)**, spread
from 2017-03-12 to 2017-09-06, the target changes on every row instead.

**Evidence that these values are linear interpolations toward the next hour:**

| Check | Interpolated hours (n = 310) | Regular hours (n = 3,785) |
|---|---|---|
| Std. dev. of row-to-row steps within the hour | max $5.0 \times 10^{-11}$ (constant step) | — |
| \|last value of hour *h* − first value of hour *h*+1\| (median) | 0.0008 | 0.23 |

- **Shape:** the step between consecutive rows is constant within each hour, i.e.
  the values lie on a straight line.
- **Destination:** each ramp ends at the value of the next hour, roughly 300× closer
  than the typical hour-to-hour jump between real assays.
- **Blocks:** interpolated hours appear in consecutive runs (e.g. a single straight
  ramp spanning three hours around 2017-05-16 19:00), consistent with missing lab
  results being filled by connecting the last known assay to the next one.

**Implication:** in these hours the target is not a measurement. It was built using
the *next* lab result, which did not exist at prediction time. Training or evaluating
on these rows leaks future information, and chronological splits do not catch it
because the leak is inside each row. How to handle them (exclude or flag) is decided
in section 1.6, together with the target definition.

In [ ]:
feed = "% Iron Feed"

hourly_feed = df.groupby("date")[feed].first().asfreq("h")
block_id = hourly_feed.ne(hourly_feed.shift()).cumsum()


runs = hourly_feed.groupby(block_id.values).agg(
    start=lambda s: s.index.min(),
    hours="size",
    value="first",
)
runs = runs[runs["value"].notna()]

print(f"blocks: {len(runs)}")
print(runs["hours"].describe())
print(runs["hours"].value_counts().sort_index().head(15))


To be analysed in next phases as a feature: “hours since last feed updating"

In [ ]:
print(runs.loc[runs["hours"].isin([6, 9]), "start"].dt.hour.value_counts().sort_index())
print(runs[runs["hours"] > 48])